In [1]:
import pandas as pd

# pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
# pd.set_option('display.max_rows', None)

# 1-1. 데이터 불러오기
# 집
train = pd.read_parquet(r'C:\Users\soohok402\내 드라이브\Github\AI_Data_exhibition\parqueted_data\train_split')
val = pd.read_parquet(r'C:\Users\soohok402\내 드라이브\Github\AI_Data_exhibition\parqueted_data\val_split')

# 회사
# train = pd.read_parquet(r'C:\Users\20162180\AI_Data_exhibition\parqueted_data\train_split')
# val = pd.read_parquet(r'C:\Users\20162180\AI_Data_exhibition\parqueted_data\val_split')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
# 2-1 ablation 대상 컬럼군 설정
# 수위 컬럼군
wl_cols = ['wl_t',
          'wl_t_minus_1h',
          'wl_t_minus_3h',
          'wl_t_minus_6h',
          'wl_t_minus_12h',
          'wl_t_minus_24h',
          'target_maxrise_6h',  # 모델링 학습부터는 반드시 빼내야 함
]

# 강수 컬럼군
rain_cols = [
            # IMERG강우
            'imerg_rain_1h',
            'imerg_rain_6h',
            'imerg_rain_24h',
            # 지상강우
            'sfc_rain_1h',
            'sfc_rain_6h',
            'sfc_rain_24h',
            # 레이더 반사도
            'radar_reflectivity_mean',
            'radar_reflectivity_p95',
            'radar_reflectivity_ge20_fraction'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

no_wl_cols = (train
              .select_dtypes(include = 'number')
              .columns
              .drop(wl_cols)  # 종속변수 여기서 제거됨
)

no_rain_cols = (train
                .select_dtypes(include = 'number')
                .columns
                .drop(rain_cols + ['target_maxrise_6h'])  # 종속변수 제거
)

no_env_cols = (train
               .select_dtypes(include = 'number')
               .columns
               .drop(env_cols + ['target_maxrise_6h'])  # 종속변수 제거
)


datasets_ablation = {
    'no_wl_cols' : no_wl_cols,
    'no_rain_cols' : no_rain_cols,
    'no_env_cols' : no_env_cols
}

print(datasets_ablation.items())  # datasets의 이름과 내용물이 같이 나옴(반환값 두개)

dict_items([('no_wl_cols', Index(['imerg_rain_1h', 'imerg_rain_6h', 'imerg_rain_24h', 'sfc_rain_1h',
       'sfc_rain_6h', 'sfc_rain_24h', 'radar_reflectivity_mean',
       'radar_reflectivity_p95', 'radar_reflectivity_ge20_fraction',
       'cat_area', 'wsarea', 'mean_elevation', 'mean_slope', 'urban_ratio',
       'forest_ratio', 'stream_order', 'event_id'],
      dtype='object')), ('no_rain_cols', Index(['wl_t', 'wl_t_minus_1h', 'wl_t_minus_3h', 'wl_t_minus_6h',
       'wl_t_minus_12h', 'wl_t_minus_24h', 'cat_area', 'wsarea',
       'mean_elevation', 'mean_slope', 'urban_ratio', 'forest_ratio',
       'stream_order', 'event_id'],
      dtype='object')), ('no_env_cols', Index(['wl_t', 'wl_t_minus_1h', 'wl_t_minus_3h', 'wl_t_minus_6h',
       'wl_t_minus_12h', 'wl_t_minus_24h', 'imerg_rain_1h', 'imerg_rain_6h',
       'imerg_rain_24h', 'sfc_rain_1h', 'sfc_rain_6h', 'sfc_rain_24h',
       'radar_reflectivity_mean', 'radar_reflectivity_p95',
       'radar_reflectivity_ge20_fraction', 'e

In [3]:
# 2-2 종속변수 설정
y_train = train['target_maxrise_6h']
y_val = val['target_maxrise_6h']

In [4]:
# 2-3. OHE station_id OHE 포함 모델링 데이터셋
# 포함한게 조금이나마 나은 결과, 포함해서 모델링 시행

# 원핫인코딩

from sklearn.preprocessing import OneHotEncoder

# 원핫인코더 생성
ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train[['station_id']])
val_st_id_ohe = ohe.transform(val[['station_id']])



In [5]:
# 3. 하나씩 모델링
# ohe와 합치기~모델링~평가 및 검증까지 for 문안에 한번에 만들기

from scipy.sparse import hstack
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error

# 해당 데이터셋 ohe와 합치기
result = {}  # 조건별 rmse 저장용 딕셔너리

for name, cols in datasets_ablation.items(): # name: 변수군의 이름, cols: 컬럼군
    print(name)
    X_train = hstack([
        train[cols],
        train_st_id_ohe
    ]).tocsr()
    
    X_val = hstack([
        val[cols],
        val_st_id_ohe
    ]).tocsr()

    # 모델링
    model = LGBMRegressor(
        random_state = 42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    # 성능평가
    rmse = root_mean_squared_error(y_val, pred)

    # 결과에 더하기
    result[name] = rmse  # result는 딕셔너리, []는 key가 됨

print(result)
    
    

no_wl_cols
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.050316 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5029
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 482
[LightGBM] [Info] Start training from score 0.111701
no_rain_cols
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.045956 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4301
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 479
[LightGBM] [Info] Start training from score 0.111701
no_env_cols
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013058 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4817
[LightGBM] [Info